# 做出第一台视频模型

我们先保留一个最笨的基线，再选择视频表示，最后接起 `frame → token → action-conditioned next token → frame`。本实验对应离散 AR 路线；连续 latent 会在第二份 Notebook 做小型对照。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.video import (
    TinyVQVAE, ActionTokenTransformer, motion_direction_accuracy,
    red_centers, video_batch_from_episodes, token_accuracy,
)
torch.manual_seed(0)


## 1. 先把时间接对

第 `t` 个按键造成 `frame[t] → frame[t+1]`。把动作错开一格，模型会学到一套看似收敛却无法控制的规律。

In [ ]:
episodes = make_pixelworld_dataset(num_episodes=8, length=8, seed=0)
current, actions, following = video_batch_from_episodes(episodes)
print('current/action/following:', tuple(current.shape), tuple(actions.shape), tuple(following.shape))
assert len(current) == len(actions) == len(following)


## 2. 复制上一帧能有多好

相邻帧本来就很像，所以复制最后一帧可能得到不错的像素分数。它完全不读动作，也不会让物体移动。这里保留它作为后续模型的下限。

In [ ]:
copy_mse = torch.nn.functional.mse_loss(current, following)
copy_direction = motion_direction_accuracy(current, current, following)
print('复制帧 MSE:', round(float(copy_mse), 5))
print('复制帧方向准确率:', copy_direction, '（stay 不能解释真实移动）')


## 3. 选择离散 token 表示

像素最直接，连续 latent 适合去噪，语义特征适合只保留任务信息。本实验选择 VQ token，是为了让下一步使用交叉熵做离散 AR。先预热普通 Autoencoder，再打开码本，避免小数据全部挤进同一个码字。

In [ ]:
images = torch.cat((current, following))
tokenizer = TinyVQVAE(codebook_size=16, embedding_size=8)
optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)
for _ in range(30):
    optimizer.zero_grad(); loss, _ = tokenizer.continuous_loss(images); loss.backward(); optimizer.step()
tokenizer.initialize_codebook(images)
print('AE warm-up loss:', round(float(loss.detach()), 4))


## 4. 打开 VQ 与 STE

前向真的选择最近码字；反向用 STE 把 Decoder 的梯度近似送回 Encoder。这里同时检查重建和码本使用数。

In [ ]:
optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-4)
losses = []
for _ in range(20):
    optimizer.zero_grad(); output = tokenizer(images); output['loss'].backward(); optimizer.step(); losses.append(float(output['loss'].detach()))
used_codes = torch.unique(output['tokens']).numel()
print('VQ loss:', round(losses[0], 4), '→', round(losses[-1], 4), 'used codes:', used_codes)
assert used_codes > 1


## 5. 用当前 token 与动作预测下一 token

一张 `16×16` 图片现在只剩 `4×4=16` 个编号。这里使用最简单的 additive action injection；第二份 Notebook 再与 no-action 和 FiLM 公平比较。

In [ ]:
with torch.no_grad():
    current_tokens = tokenizer.encode_tokens(current).flatten(1)
    next_tokens = tokenizer.encode_tokens(following).flatten(1)
dynamics = ActionTokenTransformer(codebook_size=16, model_size=32)
optimizer = torch.optim.Adam(dynamics.parameters(), lr=3e-3)
losses = []
for _ in range(35):
    optimizer.zero_grad(); loss = dynamics.loss(current_tokens, actions, next_tokens); loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
with torch.no_grad():
    logits = dynamics(current_tokens, actions)
    predicted_tokens = logits.argmax(dim=-1)
    predicted_frames = tokenizer.decode_tokens(predicted_tokens.reshape(-1, 4, 4))
accuracy = token_accuracy(logits, next_tokens)
direction = motion_direction_accuracy(current, predicted_frames, following)
print('token loss:', round(losses[0], 3), '→', round(losses[-1], 3))
print('token accuracy:', round(float(accuracy), 3),
      'decoded motion direction:', round(float(direction), 3))
print('predicted/true center:', red_centers(predicted_frames[:1]),
      red_centers(following[:1]))
assert losses[-1] < losses[0]
assert torch.isfinite(direction)


## 小结

我们先保留复制帧基线，再选择离散表示，最后用 additive 动作条件预测下一组 token。训练准确率只说明这批小数据可拟合；第二份 Notebook 会固定同一画面更换动作，并让模型连续吃自己的输出。